# Mixed-defense action-set demo

This notebook demonstrates setup, execution of the script-equivalent pipeline, and interpretation of saved artifacts.

In [1]:
from pathlib import Path
import json
import pandas as pd

from mpmgame import (
    MixedDefenseWeights,
    defense_success_sets,
    evaluate_mixed_defense_subset,
    paper_example_data,
    select_defense_subset_greedy,
)


C:\Users\perry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Setup
Load the paper-example game and compute defense success sets `S(∇)`.


In [2]:
data = paper_example_data()
success_sets = defense_success_sets(data.M, data.attacks, data.defenses)
success_sets


{'∇1': {'Δ2', 'Δ3'}, '∇2': {'Δ2'}, '∇3': {'Δ1', 'Δ3'}, '∇4': {'Δ2', 'Δ3'}}

## 2) Script-equivalent pipeline
Run greedy selection under a cardinality cap and evaluate the reduced game.


In [3]:
weights = MixedDefenseWeights(alpha_union=1.0, alpha_intersection=0.5, alpha_cardinality=0.2)
selection = select_defense_subset_greedy(success_sets, z=2, weights=weights)
evaluation = evaluate_mixed_defense_subset(data.M, data.attacks, data.defenses, selection)
selection, evaluation.value


(MixedDefenseSelection(method='greedy', selected_labels=['∇1', '∇3'], score=3.1, components=MixedDefenseObjectiveComponents(pair_union=3.0, pair_intersection=1.0, cardinality_penalty=0.4)),
 0.5)

You can reproduce full sweeps by running the benchmark script:

```bash
python scripts/mixed_defense_benchmark.py
```


## 3) Interpret saved CSV/JSON and figures
Read sweep tables and summary JSON generated in `results/mixed_defense/`.


In [6]:
results_dir = Path('../results/mixed_defense')
reports_dir = Path('reports/mixed_defense')
df = pd.read_csv(results_dir / 'sweep_results.csv')
summary = json.loads((results_dir / 'summary.json').read_text())
df.head(), summary['full_game_value']


(   method  z  weights_id  alpha_union  alpha_intersection  alpha_cardinality  \
 0  greedy  1           0          1.0                 0.0                0.0   
 1  random  1           0          1.0                 0.0                0.0   
 2  greedy  2           0          1.0                 0.0                0.0   
 3  random  2           0          1.0                 0.0                0.0   
 4  greedy  3           0          1.0                 0.0                0.0   
 
   selected_labels  selected_size  objective_score  pair_union  \
 0              ∇1              1              2.0         2.0   
 1              ∇1              1              2.0         2.0   
 2           ∇1,∇3              2              3.0         3.0   
 3           ∇2,∇3              2              3.0         3.0   
 4        ∇1,∇3,∇2              3              8.0         8.0   
 
    pair_intersection  cardinality_penalty  union_coverage  reduced_value  \
 0                2.0                

In [7]:
list(reports_dir.glob('*.png'))


[]